# Session 14 — Quantum Sinkhorn

**Author:** PPSP lab  **·**  **Date:** 2026-05-27  **·**  **Project:** Quantum Optimal Transport course  **·**  **Status:** 🔬 Teaching

## Purpose

S14 closes the M2 ↔ M3 ↔ M4 loop. We:

1. Add a **von Neumann entropy** regulariser to the QOT SDP of S13 — the operator
   analogue of S10's classical Sinkhorn entropic problem.
2. Introduce the **matrix-exponential Gibbs kernel** $K = \exp(-C/\varepsilon)$ (the
   quantum counterpart of the entrywise $K_{ij} = e^{-C_{ij}/\varepsilon}$).
3. Show that the **Amari KL-projection identity survives the lift**: the entropic-QOT
   plan is the **Umegaki relative-entropy projection** of $K$ onto the coupling
   polytope. M2 (Umegaki, S7) and M4 (couplings, S13) collapse into a single object.
4. Sweep $\varepsilon$ and watch the quantum plan blur from the sharp SDP optimum
   toward $\rho_A \otimes \rho_B$ — same trade-off as classical, lifted to operators.

## 0. What you'll be able to do

- Build the **matrix-exponential Gibbs kernel** $K = e^{-C/\varepsilon}$ and recognise it as the operator analogue of $K_{ij} = e^{-C_{ij}/\varepsilon}$.
- Solve the **entropic QOT SDP** via cvxpy's `von_neumann_entr` atom.
- Compare the entropic plan to the non-regularised SDP at small $\varepsilon$ and to $\rho_A \otimes \rho_B$ at large $\varepsilon$.
- **Verify Amari's quantum bridge** numerically: $\varepsilon\,S_{\mathrm{Umegaki}}(P_\varepsilon \| K) = \mathrm{tr}(C\,P_\varepsilon) - \varepsilon\,S(P_\varepsilon)$.
- Reason about the ε trade-off in operator land.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.quantum.composite import tensor
from qot_course.quantum.density import (
    density_matrix, maximally_mixed, von_neumann_entropy,
)
from qot_course.quantum.states import KET_PLUS
from qot_course.quantum_ot.sdp import (
    quadratic_position_cost, quantum_ot_sdp,
)
from qot_course.quantum_ot.sinkhorn import (
    matrix_gibbs_kernel, quantum_sinkhorn_sdp,
    quantum_sinkhorn_cost, umegaki_kl_to_kernel,
)

np.random.seed(42)
viz.use_course_style()

## 1. The entropic QOT problem

Add a von Neumann entropy bonus to the SDP cost:

$$
\boxed{\;P^\star_\varepsilon \,=\, \arg\min_{\rho_{AB} \in \Pi(\rho_A, \rho_B)}\
       \mathrm{tr}(C\,\rho_{AB}) - \varepsilon\,S(\rho_{AB})\;}
$$

where $S(\rho_{AB}) = -\mathrm{tr}(\rho_{AB}\log\rho_{AB})$. Three reasons (familiar from S10):

- **Strict convexity** ($S$ strictly concave on the PSD cone) ⇒ unique minimiser, smooth in inputs.
- **A Gibbs-style closed form** for the plan's structure (matrix exponential).
- **Faster, more stable solvers** in practice (cvxpy's interior point methods are well-conditioned).

We solve it directly via cvxpy's `von_neumann_entr` atom — pedagogically equivalent to
running the explicit operator Sinkhorn iterations of Peyré–Cuturi tensor fields, but
much shorter to write.

## 2. The matrix-exponential Gibbs kernel

In S10 the classical Gibbs kernel was the **entrywise** exponential
$K_{ij} = e^{-C_{ij}/\varepsilon}$. Lifting to operators, the natural analogue is the
**matrix exponential** $K = \exp(-C/\varepsilon)$. For a diagonal cost (commuting case)
the two coincide; in general the matrix exponential introduces operator mixing.

In [ ]:
C = quadratic_position_cost([0.0, 1.0])
print("cost operator C (quadratic position):")
print(np.round(C.real, 4))
print()

eps = 0.5
K = matrix_gibbs_kernel(C, eps)
print(f"matrix-exponential Gibbs kernel K = expm(-C / {eps}):")
print(np.round(K.real, 4))
print()
print(f"K Hermitian?  {np.allclose(K, K.conj().T)}")
print(f"K positive definite?  {np.min(np.linalg.eigvalsh(0.5 * (K + K.conj().T))) > 0}")
print()
print(f"log(K) recovers -C/eps?  "
      f"{np.allclose(np.linalg.eigvalsh(np.linalg.eigh(K)[1] @ np.diag(np.log(np.linalg.eigvalsh(K))) @ np.linalg.eigh(K)[1].conj().T), -np.linalg.eigvalsh(C/eps))}")

**Read the output.** $K$ is a Hermitian PSD matrix with the same eigenvectors as $C$
(since $C$ is here already diagonal). For non-diagonal $C$ the matrix exponential
mixes operators non-trivially — exactly the new content of the quantum lift.

## 3. Solving the entropic SDP

The 5-line cvxpy implementation from S13 with one extra term (`- eps * cp.von_neumann_entr(plan)`):

In [ ]:
rho_a = density_matrix(KET_PLUS)
rho_b = maximally_mixed(2)

eps = 0.5
value, plan_eps = quantum_sinkhorn_sdp(rho_a, rho_b, C, epsilon=eps)
print(f"entropic SDP objective (tr(C P) - eps * S(P)) = {value:.4f}")
print()

# Decompose: transport cost vs entropy bonus.
transport = float(np.real(np.trace(C @ plan_eps)))
eigs_plan = np.linalg.eigvalsh(0.5 * (plan_eps + plan_eps.conj().T))
eigs_plan_pos = eigs_plan[eigs_plan > 1e-12]
S_plan = float(-np.sum(eigs_plan_pos * np.log(eigs_plan_pos)))
print(f"transport part  tr(C * P_eps)  = {transport:.4f}")
print(f"entropy part    S(P_eps)        = {S_plan:.4f} nats  ({S_plan/np.log(2):.4f} bits)")
print(f"eps * S(P_eps)                  = {eps * S_plan:.4f}")
print(f"objective check: transport - eps*S = {transport - eps * S_plan:.4f}   (matches value above)")

**Read the output.** The entropic SDP returns an *objective value* (cost minus
$\varepsilon$ times entropy), not the transport cost itself. To compare against the
unregularised QOT we extract the transport part separately. As $\varepsilon$ shrinks,
the entropy term becomes negligible and the transport cost approaches the LP/SDP value.

## 4. The ε trade-off — operator version of S10's classical sweep

We compute the entropic plan at four $\varepsilon$ values and watch the structure
sweep from the *sharp* non-regularised SDP plan to the *product coupling*
$\rho_A \otimes \rho_B$.

In [ ]:
epsilons = [0.05, 0.5, 5.0, 100.0]
plans = [quantum_sinkhorn_sdp(rho_a, rho_b, C, epsilon=eps)[1] for eps in epsilons]
product = tensor(rho_a, rho_b)

fig, axes = plt.subplots(1, len(epsilons) + 1, figsize=(15, 3.6))
for ax, plan, eps in zip(axes[:-1], plans, epsilons):
    im = ax.imshow(plan.real, cmap="RdBu_r", vmin=-0.5, vmax=0.5, origin="lower")
    ax.set_title(rf"$\varepsilon = {eps}$" + "\nRe$(\\rho^\\star_{AB})$", pad=8)
    ax.set_xticks(range(4)); ax.set_yticks(range(4)); ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.85)

im = axes[-1].imshow(product.real, cmap="RdBu_r", vmin=-0.5, vmax=0.5, origin="lower")
axes[-1].set_title("product\n$\\rho_A \\otimes \\rho_B$", pad=8)
axes[-1].set_xticks(range(4)); axes[-1].set_yticks(range(4)); axes[-1].grid(False)
fig.colorbar(im, ax=axes[-1], shrink=0.85)

fig.suptitle("Entropic QOT plans $\\rho^\\star_{AB, \\varepsilon}$: sharp SDP $\\to$ blurry independent coupling",
             fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

# How close is the large-eps plan to the product?
distance_to_product = float(np.linalg.norm(plans[-1] - product))
print(f"||P_eps=100 - rho_A otimes rho_B||_F = {distance_to_product:.4f}   (should be small)")

**Read the figure.** From left to right: $\varepsilon = 0.05$ gives the sharp SDP-style
plan; as $\varepsilon$ grows, the structure smooths out; at $\varepsilon = 100$ the plan
is essentially indistinguishable from $\rho_A \otimes \rho_B$ (rightmost panel for
reference). This is the *operator* version of the S10 sweep, where the entrywise sharpness
also collapsed toward the independent coupling at large $\varepsilon$. **Same structure,
lifted to matrices.**

## 5. Amari's quantum bridge — Umegaki KL projection

The single most beautiful equation of the course. Plug the entropic objective into the
**Umegaki** relative entropy with the matrix-exponential Gibbs kernel $K = e^{-C/\varepsilon}$:

$$ \varepsilon\,S_{\mathrm{Umegaki}}(\rho_{AB} \,\|\, K)
   \,=\, \mathrm{tr}(\rho_{AB} \log\rho_{AB}) - \mathrm{tr}(\rho_{AB} \log K) \cdot \varepsilon
   \,=\, \mathrm{tr}(C\,\rho_{AB}) - \varepsilon\,S(\rho_{AB}). $$

So minimising the entropic QOT objective over couplings is the same as minimising
$S_{\mathrm{Umegaki}}(\rho_{AB} \,\|\, K)$ over couplings. **The entropic quantum OT
plan is the Umegaki relative-entropy projection of the matrix-exponential Gibbs kernel
onto the coupling polytope.**

In one image: M2 (the *information* spine of the course — quantum KL) and M3+M4 (the
*transport* spine — quantum couplings) share a single Bregman geometry. The Sinkhorn
algorithm is the operator-Bregman iteration onto the marginal constraints. We verify
the identity numerically.

In [ ]:
# Verify the Amari quantum identity at the entropic optimum.
eps = 0.4
value, plan = quantum_sinkhorn_sdp(rho_a, rho_b, C, epsilon=eps)
K = matrix_gibbs_kernel(C, eps)

# LHS:  eps * S_Umegaki(P_eps || K)
lhs = eps * umegaki_kl_to_kernel(plan, K)

# RHS:  tr(C * P_eps) - eps * S(P_eps)
transport = float(np.real(np.trace(C @ plan)))
vals = np.linalg.eigvalsh(0.5 * (plan + plan.conj().T))
vals = vals[vals > 1e-12]
S_plan = float(-np.sum(vals * np.log(vals)))
rhs = transport - eps * S_plan

print(f"eps * S_Umegaki(P_eps || K)              = {lhs:.6f}")
print(f"tr(C * P_eps) - eps * S(P_eps)            = {rhs:.6f}")
print(f"Amari quantum bridge holds?                {np.isclose(lhs, rhs, atol=1e-4)}")
print()
print("This is M2 ∩ M3 ∩ M4 collapsing to a single equation.")

**Read the output.** The two quantities agree to numerical precision. The entropic
QOT plan satisfies the Amari-type identity: *minimising the entropic transport cost
over couplings is identical to minimising the Umegaki relative entropy to the matrix
exponential of $-C/\varepsilon$*.

This is the **most important structural observation of M4**: the *information geometry*
of S7 (Umegaki) and the *transport geometry* of S13 (couplings) are different views of
the same Bregman structure. The Sinkhorn algorithm of S10 was the *iterative Bregman
projection* in classical land; its quantum analogue (Peyré–Cuturi tensor fields,
Pelikh–Gerolin) is the *iterative operator Bregman projection* under the Umegaki
divergence.

## 6. Bringing it all together — the completed bridge

| | **Classical** (S10) | **Quantum** (S14) |
|---|---|---|
| Variable | transport plan $P_{ij} \ge 0$ | quantum coupling $\rho_{AB} \succeq 0$ |
| Entropy | Shannon $H(P) = -\sum P_{ij}\log P_{ij}$ | von Neumann $S(\rho_{AB}) = -\mathrm{tr}(\rho_{AB}\log\rho_{AB})$ |
| Gibbs kernel | entrywise $K_{ij} = e^{-C_{ij}/\varepsilon}$ | matrix exp $K = \exp(-C/\varepsilon)$ |
| Sinkhorn iteration | multiplicative row/col scaling | operator multiplicative scaling (Peyré tensor fields) |
| Bridge identity | $\varepsilon\,\mathrm{KL}(P\|K) = \langle C, P\rangle - \varepsilon H(P)$ | $\varepsilon\,S_{\mathrm{Umegaki}}(\rho_{AB}\|K) = \mathrm{tr}(C\,\rho_{AB}) - \varepsilon S(\rho_{AB})$ |
| **Reading** | **Sinkhorn = KL projection** | **Quantum Sinkhorn = Umegaki projection** |

The whole course has been about constructing this table. **Information geometry and
transport geometry are the same geometry, lifted from probability vectors to density
matrices.** The Sinkhorn algorithm in either world is the operator-Bregman iteration
under the appropriate divergence.

Applications follow this template — for instance the Peyré, Chizat, Vialard, Schmitzer
(2019) **diffusion-tensor / texture interpolation** uses entropic QOT to morph between
tensor-valued fields on images. We mention but do not implement this; the *machinery*
is exactly what we built above.

## 7. Your turn

1. **The ε bias on the transport cost.** For the running example ($|+\rangle\langle+|$
   vs $I/2$), plot $\mathrm{tr}(C\,P_\varepsilon^\star)$ as a function of $\varepsilon$
   on a log scale. The bias should scale as $\sim \varepsilon$ asymptotically (a
   first-order expansion around the unregularised optimum).
2. **Operator Sinkhorn iteration.** Implement the explicit Peyré–Cuturi iteration:
   alternately update Hermitian matrices $L_A, L_B$ such that the plan
   $\exp(L_A \otimes I + I \otimes L_B - C/\varepsilon)$ has the right marginals. Use
   Newton or fixed-point iteration on each side. Compare to the cvxpy solution above —
   they should give the same plan.
3. **Quantum Sinkhorn divergence.** Compute the debiased
   $S_\varepsilon(\rho_A, \rho_B) = \mathrm{QOT}_\varepsilon(\rho_A, \rho_B)
   - \tfrac12 \mathrm{QOT}_\varepsilon(\rho_A, \rho_A) - \tfrac12 \mathrm{QOT}_\varepsilon(\rho_B, \rho_B)$
   for two qubit states and check that $S_\varepsilon(\rho_A, \rho_A) = 0$ and
   $S_\varepsilon(\rho_A, \rho_B) > 0$.

## Conclusions & references

- **Quantum Sinkhorn** = the entropic QOT SDP with a von Neumann entropy regulariser. Five extra characters of cvxpy code.
- **Matrix-exponential Gibbs kernel** $K = \exp(-C/\varepsilon)$ is the operator analogue of $K_{ij} = e^{-C_{ij}/\varepsilon}$.
- **Amari's quantum bridge.** $\varepsilon S_{\mathrm{Umegaki}}(P_\varepsilon \| K) = \mathrm{tr}(C\,P_\varepsilon) - \varepsilon S(P_\varepsilon)$. The quantum Sinkhorn plan is the Umegaki KL projection of $K$ onto the coupling polytope.
- The **ε trade-off** is the same as classical: small $\varepsilon$ ⇒ sharp SDP solution, large $\varepsilon$ ⇒ blurry product coupling $\rho_A \otimes \rho_B$.
- **M2 + M3 + M4 close into a single Bregman geometry.** Information geometry (KL / Umegaki) and transport geometry (couplings) are the same geometry, lifted.
- **Next (S15 — Capstone):** an open research problem on a *synthetic Kuramoto dyad* with known injected coupling. Compare QOT-based coupling vs Umegaki-based vs PLV / Euclidean baselines.

**References:** M. Cuturi, "Sinkhorn distances", NeurIPS (2013); G. Peyré, L. Chizat,
F.-X. Vialard, B. Schmitzer, "Quantum entropic regularization of matrix-valued optimal
transport", *European J. Appl. Math.* **30**, 1079 (2019); D. Trevisan, *Optimal
transport methods for quantum systems* (arXiv:2202.02091, 2022); A. Gerolin, M. Pelikh,
*Quantum optimal transport: an invitation* (2024 lecture notes); S.-i. Amari,
*Information Geometry and Its Applications* (Springer, 2016), sec. 7.5. Previous:
`notebooks/04_QuantumOptimalTransport/02_sdp.ipynb`.